# 04 Infer Cluster ADS
Bestes NAND-Modell laden, ADS-Pairs scoren, DBSCAN+Constraints anwenden, Exporte schreiben.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import sys

RUN_STAGE = "smoke"
RUN_ID = "replace_with_run_id_from_00"
DEVICE = "auto"
def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "configs").exists():
            return candidate.resolve()
    return start.resolve()

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.common.config import load_yaml
from src.common.io_schema import read_parquet
from src.approaches.nand.infer_pairs import score_pairs_with_checkpoint
from src.approaches.nand.cluster import cluster_blockwise_dbscan
from src.approaches.nand.export import build_publication_author_mapping

cluster_cfg = load_yaml(PROJECT_ROOT / "configs/clustering/dbscan_paper.yaml")

subset_dir = PROJECT_ROOT / "data/subsets/cache" / RUN_ID
emb_dir = PROJECT_ROOT / "artifacts/embeddings" / RUN_ID
ckpt_dir = PROJECT_ROOT / "artifacts/checkpoints" / RUN_ID
pair_score_dir = PROJECT_ROOT / "artifacts/pair_scores" / RUN_ID
cluster_dir = PROJECT_ROOT / "artifacts/clusters" / RUN_ID
metrics_dir = PROJECT_ROOT / "artifacts/metrics" / RUN_ID
pair_score_dir.mkdir(parents=True, exist_ok=True)
cluster_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

ads_mentions = read_parquet(subset_dir / f"ads_mentions_{RUN_STAGE}.parquet")
ads_pairs = read_parquet(subset_dir / f"ads_pairs_{RUN_STAGE}.parquet")
ads_chars = np.load(emb_dir / f"ads_chars2vec_{RUN_STAGE}.npy")
ads_text = np.load(emb_dir / f"ads_specter_{RUN_STAGE}.npy")

with (metrics_dir / "03_train_manifest.json").open("r", encoding="utf-8") as f:
    train_manifest = json.load(f)

best_checkpoint = train_manifest["best_checkpoint"]
best_threshold = train_manifest["best_threshold"]
print("Best checkpoint:", best_checkpoint)
print("Best threshold:", best_threshold)


In [ ]:
pair_scores_path = pair_score_dir / f"ads_pair_scores_{RUN_STAGE}.parquet"
pair_scores = score_pairs_with_checkpoint(
    mentions=ads_mentions,
    pairs=ads_pairs,
    chars2vec=ads_chars,
    text_emb=ads_text,
    checkpoint_path=best_checkpoint,
    output_path=pair_scores_path,
    device=DEVICE,
)
print("Pair scores:", len(pair_scores), "->", pair_scores_path)


In [ ]:
clusters_path = cluster_dir / f"ads_clusters_{RUN_STAGE}.parquet"
clusters = cluster_blockwise_dbscan(
    mentions=ads_mentions,
    pair_scores=pair_scores,
    cluster_config=cluster_cfg,
    output_path=clusters_path,
)
print("Clusters:", len(clusters), "->", clusters_path)


In [ ]:
# Cluster-QC
merged = ads_mentions[["mention_id", "block_key"]].merge(clusters, on=["mention_id", "block_key"], how="left")
cluster_size = merged.groupby(["block_key", "author_uid"]).size().rename("size").reset_index()

singleton_ratio = float((cluster_size["size"] == 1).mean()) if len(cluster_size) else 0.0
print("Singleton ratio:", singleton_ratio)
print("Cluster size describe:")
print(cluster_size["size"].describe())

unstable_blocks = cluster_size.groupby("block_key")["size"].agg(["count", "max"]).sort_values(["count","max"], ascending=False).head(20)
unstable_blocks


In [ ]:
# Exporte
mention_export = cluster_dir / f"mention_author_uid_{RUN_STAGE}.parquet"
pub_export = cluster_dir / f"publication_authors_{RUN_STAGE}.parquet"

clusters.to_parquet(mention_export, index=False)
pub_map = build_publication_author_mapping(mentions=ads_mentions, clusters=clusters, output_path=pub_export)

print("mention export:", mention_export)
print("publication export:", pub_export)
print("publication rows:", len(pub_map))
